In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import linalg as LA

from ipywidgets import interact, widgets, interact_manual

In [ ]:
class SchrodingerEquationSolver(object):
  """Solves Schrodinger equation 
  for zero-order 1D potentials
  using Numerov scheme and plots its solutions

  Args:
    m (float): Particle mass
    Ngrid (int): Number of grid points
    xmin (float): Minimum grid point
    xmax (float): Maximum grid point"""
  def __init__(self, m, Ngrid, xmin, xmax):
    print('In SchrodingerEquationSolver.__init__()')
    self.hbar = 1
    self.m = m
    self.Ngrid = Ngrid
    self.xmin = xmin
    self.xmax = xmax

    #A vector spanning from xmin to xmax with Ngrid grid points
    self.xvec = np.linspace(self.xmin, self.xmax, self.Ngrid)
    self.Vx = 0
    
    # Initialise kinetic and potential energy matrices
    self.Ekin = np.zeros(self.Ngrid)
    self.Epot = np.zeros(self.Ngrid)

  def plotPotential(self):
    """Plot potential on the x-axis"""
    f, ax = plt.subplots()
    ax.plot(self.xvec, self.Vx)
    ax.set_xlabel('position $x$')
    ax.set_ylabel('potential $V(x)$')
    ax.set(title = 'Potential')

  def calculateKineticEnergy(self):
    """Calculate the diagonalised kinetic energy matrix 
    using the Numerov scheme"""
    #resolution of the grid
    dx = np.diff(self.xvec).mean()


    dia = -2*np.ones(self.Ngrid)
    offdia = np.ones(self.Ngrid-1)
    d2grid = np.mat(
        np.diag(dia,0) 
        + np.diag(offdia,-1) 
        + np.diag(offdia,1)
      )/dx**2
    
    # Avoid strange things at the edge of the grid
    d2grid[0,:]=0;
    d2grid[self.Ngrid-1,:]=0

    return -self.hbar**2 / (2 * self.m) * d2grid

  def calculatePotentialEnergy(self):
    """Calculate the diagonalised potential energy matrix
    from the 1D potential vector"""
    return np.mat(np.diag(self.Vx,0))

  def calculateHamiltonian(self):
    """Combine the diagonalised kinetic and potential energies 
    to a diagonalised Hamiltonian"""
    self.Ekin = self.calculateKineticEnergy()
    self.Epot = self.calculatePotentialEnergy()

    self.H = self.Ekin + self.Epot

  def calculateEigenvaluesAndEigenvectors(self):
    """Calculate the sorted eigenvalues (energies) 
    and eigenvectors (wavefunctions) 
    of the diagonalised Hamiltonian"""
    # diagonalization
    w, v = LA.eig(self.H)
    # sort it such that things look nice later
    sortinds = np.argsort(w)
    self.HamiltonianEigenvectors = v[:,sortinds]
    self.HamiltonianEigenvalues = w[sortinds]
    
  
  def plotEnergies(self, nMin, nMax):
    """Plot the Hamiltonian eigenvalues (energies)
    for a given range of quantum numbers

    Args:
      nMin (int): Minimum quantum number
      nMax (int): Maximum quantm number"""
    f, ax = plt.subplots()
    iMin = max(nMin, 0)
    iMax = min(nMax, len(self.HamiltonianEigenvalues))
    ax.plot(self.HamiltonianEigenvalues[iMin:iMax],'o')
    ax.set_ylabel('Energy')
    ax.set_xlabel('index $n$')
    ax.set_yticks(
      np.arange(
        self.HamiltonianEigenvalues[iMin], 
        self.HamiltonianEigenvalues[iMax], 
        0.5
      )
    )

  def plotWaveFunction(self, n): 
    """Plot wavefunction for a given quantum number
    
    Args:
      n: quantum number"""
    f, ax = plt.subplots()
    ax.plot(
      self.xvec,np.real(
        self.HamiltonianEigenvectors[:,n] / self.maximumAmplitude
      )
    )
    ax.set(title='Eigenfunction for n=%d'%(n),xlabel='x')
    
  def plotProbabilityDensity(self, n):
    """Plot probability density for a given quantum number
    
    Args:
      n: quantum number"""
    f, ax = plt.subplots()
    ax.plot(
        self.xvec,
        np.power(
          np.abs(
            self.HamiltonianEigenvectors[:,n]),
            2
          ) / self.maximumAmplitude**2
        ) 
    ax.set(title='Probability Density Function for n=%d'%(n),xlabel='x')

  def plotSolutions(
      self,
      nMin, 
      n, 
      nMax,
  ):
    """Plot Schrodinger equation solutions: 
      Hamiltonian eigenvalues (energies) 
        for a given range of quantum numbers,
      Hamiltonian eigenfunction (wavefunction) 
        for a given quantum number,
      and probability density (wavefunction squared)
      for a given quantum number

    Args:
      nMin (int): Minimum quantum number
      n (int): Quantum number 
        to plot the wavefunction and probability density at
      nMax (int): Maximum quantum number"""
    self.plotEnergies(nMin, nMax)
    self.maximumAmplitude = np.amax(self.HamiltonianEigenvectors[:, n])
    self.plotWaveFunction(n)
    self.plotProbabilityDensity(n)
  
  def solve(self):
    """Solve the Schrodinger equation 
    and plot its solutions using input quantum numbers"""
    self.plotPotential()
    self.calculateHamiltonian()
    self.calculateEigenvaluesAndEigenvectors()

    interact_manual(self.plotSolutions,
      nMin = widgets.IntText(),
      n=widgets.IntText(),
      nMax = widgets.IntText()
    )